In [76]:
from math import sin, cos, asin, acos, pi, sqrt, radians, degrees
import numpy as np
import sympy as sp
from sympy import lambdify

### Gravity Compensation at Aerial Phase

In this phase, the leg can be treated as moving in free space with the relatively static base located at the hip joint. The joints need to hold the linkages in position by "pulling them up" and preventing them from falling due to gravity.

We first calculate and verify FK with the base frame located at the hip joint (joint 1).

In [77]:
# FK calculations
l1 = 0.125
l2 = 0.215
lc1 = 0.0613
lc2 = 0.1075
th = radians(77.52)   # Angle of link 1 CoM WRT the "imaginary" link 1

q1 = -129   # WRT to the +x axis, CCW positive. THIS IS DIFFERENT FROM MOTOR JOINT ANGLE BY pi/2
q2 = 100    # WRT to the extension of link 1, CCW positive.

q1 = radians(q1)
q2 = radians(q2) 

P_ca = np.array([l2*cos(q1 + q2) + l1*cos(q1), 
                 l2*sin(q1 + q2) + l1*sin(q1)])
# print("Foot contact WRT joint 1", P_ca)

Calculate the Jacobians of link CoMs using sumpy. Note that the CoM of link 1 is placed with an offset angle, theta.

<img src="Gravity_aerial.jpg" alt="Joint angles" width="50%"/>

The gravity is acting in the -Z direction. We calculate the gravity compensation torque using the equation: 

$$
\tau_g = \sum_{i=1}^{n} \left( J_{\text{com},i}^\top(\mathbf{q}) \right) (m_i \mathbf{g})
$$

In [78]:
q1, q2, l1, lc1, lc2, g, m1, m2, th = sp.symbols('q1 q2 l1 lc1 lc2 g m1 m2 th')
pc1 = sp.Matrix([
    lc1 * sp.cos(q1 + th),
    lc1 * sp.sin(q1 + th)
])
pc2 = sp.Matrix([
    l1 * sp.cos(q1) + lc2 * sp.cos(q1 + q2), 
    l1 * sp.sin(q1) + lc2 * sp.sin(q1 + q2)
])

q = sp.Matrix([q1, q2])
Jc1 = pc1.jacobian(q)
Jc2 = pc2.jacobian(q)
# print("Jc1: ")
# sp.pprint(Jc1)
# print("Jc2: ")
# sp.pprint(Jc2)

g0 = sp.Matrix([0, -g])
# print("Tau: ")
Tau = Jc1.T * m1 * g0 + Jc2.T * m2 * g0
# sp.pprint(Tau)

We can try to verify our expression by plugging in some real values. We keep m1, m2 and g as symbolic. Note that this is how each joint experiences gravity. To counteract, a negative sign needs to be added.

In [79]:
f = lambdify((q1, q2, th, l1, lc1, lc2), Tau, modules='numpy')
l1 = 0.125
l2 = 0.215
lc1 = 0.0613
lc2 = 0.1075
th = radians(77.52)   # Angle of link 1 CoM WRT the "imaginary" link 1

q1 = radians(-129)   # WRT to the +x axis, CCW positive. THIS IS DIFFERENT FROM MOTOR JOINT ANGLE BY pi/2
q2 = radians(100)    # WRT to the extension of link 1, CCW positive.

result = f(q1, q2, th, l1, lc1, lc2)
print(result)

[[-0.0381768909439213*g*m1 - 0.0153565696362554*g*m2]
 [-0.0940216185174851*g*m2]]


### Gravity Compensation at Stance Phase

In this phase, the leg is mounted to the gantry and is in contact with the ground via its leg tip. The gantry permits +/-Z movements but restricts +/-X movements.

We first calculate and verify FK with the base frame located at the foot. With the switch of base frame, we need to define a new set of joint angles which are linked to the original joint values.

In [80]:
# FK calculations
l1 = 0.125
l2 = 0.215
lc1 = 0.0613
lc2 = 0.1075
th = radians(77.52)   # Angle of link 1 CoM WRT the "imaginary" link 1

q1 = -129   # WRT to the +x axis, CCW positive. THIS IS DIFFERENT FROM MOTOR JOINT ANGLE BY pi/2
q2 = 100    # WRT to the extension of link 1, CCW positive.

q1 = radians(q1)
q2 = radians(q2) 

# When base is at foot contact
q1p = q2 + (q1 + pi)
q2p = - q2
# print(f"q1p = {degrees(q1p)}, q2p = {degrees(q2p)}")
P_ac = np.array([l1*cos(q1p + q2p) + l2*cos(q1p), 
                 l1*sin(q1p + q2p) + l2*sin(q1p)])
# print("Joint 1 WRT foot contact", P_ac)

Calculate the CoM positions of link 2 and link 1. We will need to calculate some new intermediate falues now that we switched the base frame. For Link2, we simply subtract lc2 from l2 to obtain lc2p. For Link1, the law of cosines is used to obtain lc1p due to the CoM angle offset.

<img src="Gravity_stance.jpg" alt="Joint angles" width="50%"/>

In [81]:
lc2p = l2 - lc2

lc1p = sqrt(l1**2 + lc1**2 - 2*l1*lc1*cos(th))
thp = acos((l1**2 + lc1p**2 - lc1**2) / (2*l1*lc1p))
print(f"lc2p = {lc2p}, lc1p = {lc1p}, thp = {degrees(thp)}")

P_2c = np.array([lc2p * cos(q1p),
                 lc2p * sin(q1p)])
P_1c = np.array([l2 * cos(q1p) + lc1p * cos(q1p + q2p - thp),
                 l2 * sin(q1p) + lc1p * sin(q1p + q2p - thp)])
print(f"P_2c = {P_2c}, P_1c = {P_1c}")


lc2p = 0.1075, lc1p = 0.12677135224324168, thp = 28.172143173505486
P_2c = [-0.09402162  0.05211703], P_1c = [-0.0712013   0.15341676]


Calculate the Jacobians of the CoM positions to the new base frame using Sympy:

In [82]:
q1p, q2p, l1, l2, lc1p, lc2p, g, m0, m1, m2, thp = sp.symbols('q1p q2p l1 l2 lc1p lc2p g m0 m1 m2 thp')

P_ac = sp.Matrix([l1 * sp.cos(q1p + q2p) + l2 * sp.cos(q1p), 
                  l1 * sp.sin(q1p + q2p) + l2 * sp.sin(q1p)])
P_2c = sp.Matrix([lc2p * sp.cos(q1p),
                  lc2p * sp.sin(q1p)])
P_1c = sp.Matrix([l2 * sp.cos(q1p) + lc1p * sp.cos(q1p + q2p - thp),
                  l2 * sp.sin(q1p) + lc1p * sp.sin(q1p + q2p - thp)])
qp = sp.Matrix([q1p, q2p])

Jca = P_ac.jacobian(qp)
Jc2 = P_2c.jacobian(qp)
Jc1 = P_1c.jacobian(qp)
print("Jca: ")
sp.pprint(Jca, use_unicode=True)
print("Jc2: ")
sp.pprint(Jc2, use_unicode=True)
print("Jc1: ")
sp.pprint(Jc1, use_unicode=True)

Jca: 
⎡-l₁⋅sin(q1p + q2p) - l₂⋅sin(q1p)  -l₁⋅sin(q1p + q2p)⎤
⎢                                                    ⎥
⎣l₁⋅cos(q1p + q2p) + l₂⋅cos(q1p)   l₁⋅cos(q1p + q2p) ⎦
Jc2: 
⎡-lc2p⋅sin(q1p)  0⎤
⎢                 ⎥
⎣lc2p⋅cos(q1p)   0⎦
Jc1: 
⎡-l₂⋅sin(q1p) - lc1p⋅sin(q1p + q2p - thp)  -lc1p⋅sin(q1p + q2p - thp)⎤
⎢                                                                    ⎥
⎣l₂⋅cos(q1p) + lc1p⋅cos(q1p + q2p - thp)   lc1p⋅cos(q1p + q2p - thp) ⎦


The gravity is acting in the -Z direction. With the new base frame, we calculate the gravity compensation torque using the equation: 
$$
\tau_g = \sum_{i=1}^{n} \left( J_{\text{com},i}^\top(\mathbf{q}) \right) (m_i \mathbf{g})
$$

In [83]:
g0 = sp.Matrix([0, -g])
print("Tau: ")
Tau = Jca.T * m0 * g0 + Jc2.T * m2 * g0 + Jc1.T * m1 * g0
print(sp.sstr(Tau))
# sp.pprint(Tau)

Tau: 
Matrix([
[-g*lc2p*m2*cos(q1p) - g*m0*(l1*cos(q1p + q2p) + l2*cos(q1p)) - g*m1*(l2*cos(q1p) + lc1p*cos(q1p + q2p - thp))],
[                                                     -g*l1*m0*cos(q1p + q2p) - g*lc1p*m1*cos(q1p + q2p - thp)]])


We can try to verify our expression by plugging in some real values. We keep m0, m1, m2 and g as symbolic.

In [84]:
f = lambdify((q1p, q2p, thp, l1, l2, lc1p, lc2p), Tau, modules='numpy')
l1 = 0.125
l2 = 0.215
lc1 = 0.0613
lc2 = 0.1075
th = radians(77.52)   # Angle of link 1 CoM WRT the "imaginary" link 1

q1 = radians(-129)   # WRT to the +x axis, CCW positive. THIS IS DIFFERENT FROM MOTOR JOINT ANGLE BY pi/2
q2 = radians(100)    # WRT to the extension of link 1, CCW positive.

q1p = q2 + (q1 + pi)
q2p = - q2
lc2p = l2 - lc2
lc1p = sqrt(l1**2 + lc1**2 - 2*l1*lc1*cos(th))
thp = acos((l1**2 + lc1p**2 - lc1**2) / (2*l1*lc1p))

result = f(q1p, q2p, thp, l1, l2, lc1p, lc2p)
print("Tau: ", result)

Tau:  [[0.10937818815374*g*m0 + 0.0712012972098192*g*m1 + 0.0940216185174851*g*m2]
 [-0.0786650488812296*g*m0 - 0.116841939825151*g*m1]]


However, the above method assumes that the foot location is an active joint that produces torque, whereas in reality it is just a passive, contact joint. Therefore, the calculated gravity compensation torque on the knee joint may not be accurate. In the following section, we attempt to calculate the knee joint's gravity compensation torque using FBD.

<img src="Gravity_FBD.jpg" alt="Gravity FBD" width="75%"/>

Based on the FBD, the joint torques required to offset gravity can be derived as follows:

$$
\tau_2 = -F_N \cdot l_2 \cdot \cos(q_1 + q_2) 
         - F_f \cdot l_2 \cdot \sin(q_1 + q_2) 
         + m_2 \cdot g \cdot l_{c2} \cdot \cos(q_1 + q_2)
$$

$$
\tau_1 = F_f \left( l_1 \sin(q_1' + q_2') + l_2 \sin(q_1') \right)
         - \left( 
             m_2 g \cdot l_{c2}' \cos(q_1') 
             + m_0 g \cdot \left( l_1 \cos(q_1' + q_2') + l_2 \cos(q_1') \right) 
             + m_1 g \cdot \left( l_{c1}' \cos(q_1' + q_2' - \theta_{hp}) + l_2 \cos(q_1') \right)
           \right)
$$
If we substitute in $F_\mathrm{N}$ and ignore the friction term at the foot contact, the equations will be reduced to:

$$
\tau_2 = -(m_0 + m_1 + m_2) \cdot g \cdot l_2 \cdot \cos(q_1 + q_2) + m_2 \cdot g \cdot l_{c2} \cdot \cos(q_1 + q_2)
$$

$$
\tau_1 = - \left( 
             m_2 g \cdot l_{c2}' \cos(q_1') 
             + m_0 g \cdot \left( l_1 \cos(q_1' + q_2') + l_2 \cos(q_1') \right) 
             + m_1 g \cdot \left( l_{c1}' \cos(q_1' + q_2' - \theta_{hp}) + l_2 \cos(q_1') \right)
           \right)
$$


In [ ]:
g, m0, m1, m2 = sp.symbols('g m0 m1 m2')

mu_k = 0  # Coefficient of kinetic friction (assumption)
F_N = (m0 + m1 + m2) * g  # Normal force at the foot contact
F_f = mu_k * F_N

Tau_2 = -F_N * l2 * cos(q1 + q2) - F_f * l2 * sin(q1 + q2) + m2 * g * lc2 * cos(q1 + q2)
Tau_1 = -F_f * (l1*sin(q1p+q2p)+l2*sin(q1p)) + (m2*g * lc2p*cos(q1p) + m0*g * (l1*cos(q1p+q2p)+l2*cos(q1p)) + m1*g * (lc1p*cos(q1p+q2p-thp)+l2*cos(q1p)))

Tau_p = sp.Matrix([Tau_1, Tau_2])
print("Tau_p: ", Tau_p)

# f_2 = lambdify((q1p, q2p, thp, l1, l2, lc1p, lc2p), Tau_p, modules='numpy')


Tau_p:  Matrix([[0.10937818815374*g*m0 + 0.0712012972098192*g*m1 + 0.0940216185174851*g*m2], [0.0940216185174851*g*m2 - 0.18804323703497*g*(m0 + m1 + m2)]])
